# Setup Env

In [3]:
# !pip install -q transformers datasets accelerate bitsandbytes sentence-transformers \
#                 langchain langchain-huggingface langchain-chroma langchain-community \
#                 --upgrade

# Fine-tune Reranking Cross-Encoder with `transformers` and Parameter-Efficient Fine-Tuning

In [4]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    )
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset, load_dataset

In [5]:
# @title Load base model and add low-rank adapters

# model_name = "PORTULAN/albertina-1b5-portuguese-ptbr-encoder"
model_name = "PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir"
model = AutoModelForSequenceClassification.from_pretrained(model_name,
                                                           num_labels=1, # 1 output neuron for relevance estimation as a binary classification problem
                                                           )
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Check number of trainable and non-trainable parameters
trainable_params = 0
all_params = 0
for _, param in model.named_parameters():
    all_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()
print(f"\ntrainable params: {trainable_params:,} || all params: {all_params:,} || trainable%: {100 * trainable_params / all_params:.2f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17727.18it/s]
BertForSequenceClassification LOAD REPORT from: PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



trainable params: 108,923,905 || all params: 108,923,905 || trainable%: 100.00


In [6]:
# @title Check model module names

model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(29794, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [7]:
# @title Add **Lo**w-**R**ank **A**dapters (LoRA) to model

# Configure adapters
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8, # Adapter rank
    lora_alpha=16, # Usually set to 2 * rank
    lora_dropout=0.1,
    target_modules=["query", "key", "value", "dense"], # Target attention and FF layers (model module names)
    modules_to_save=["classifier", "pooler"] # CRITICAL: Fully train the added classification head
    )

# Wrap the base model with the adapters
model = get_peft_model(model, peft_config)
# This is the step that block us from training/fine-tuning a model directly in `sentence-transformers`.
# `CrossEncoderTrainer` requires model of class `CrossEncoder` and not the LoRA PEFT wrapper, but `CrossEncoder`
# has no `.add_adapter` method natively like `SentenceTransformer` has.

# Print trainable parameters now. A few less parameters to optimize huh?
model.print_trainable_parameters()

trainable params: 1,918,465 || all params: 110,842,370 || trainable%: 1.7308


In [8]:
# @title Load and preprocess part of the local dataset


# Load partial dataset
dataset = load_dataset("parquet", data_dir="/content/mmarco-pt", split="train")

# Convert (query, positive, negative) to (query, context, label)
def make_pairs(batch):
    queries = []
    contexts = []
    labels = []
    # Process data in batches
    for q, pos, neg in zip(batch['query'], batch['positive'], batch['negative']):
        # Positive pair
        queries.append(q)
        contexts.append(pos)
        labels.append(1.0)
        # Negative pair
        queries.append(q)
        contexts.append(neg)
        labels.append(0.0)
    return {
        "query": queries,
        "context": contexts,
        "labels": labels
    }

print("Converting triples into pairs...")
dataset = dataset.map(
    make_pairs,
    batched=True,
    remove_columns=dataset.column_names
)

# Split and tokenize data
def preprocess_function(examples):
    # Tokenize the pairs together
    return tokenizer(
        examples["query"],
        examples["context"],
        truncation=True,
        padding="max_length",
        max_length=512,
        add_special_tokens=True, # Must be True (default value) to ensure tokenizer adds
                                 # the required separator tokens between query and context
    )

print("Tokenizing query-context pairs...")
dataset = dataset.map(preprocess_function, batched=True)

print("Train/Test-split")
dataset = dataset.train_test_split(test_size=0.1, seed=42)
dataset

EmptyDatasetError: The directory at /content/mmarco-pt doesn't contain any data files

In [ ]:
# If we hadn't used LoRA, we could still just train the uninitialized weights added by the `AutoModel` by
# freezing all other layers

# Freeze all parameters
# for param in model.parameters():
#     param.requires_grad = False

# "Unfreeze" (melt?) target parameters (`classifier` and `pooler`)
# target_layers = ["classifier", "pooler"]
# for name, param in model.named_parameters():
#     if any([target in name for target in target_layers]): # If name contains "classifier" or "pooler"
#         param.requires_grad = True


In [ ]:
# @title Setup training loop

# Define hyperparameter arguments optimized for LoRA
training_args = TrainingArguments(
    output_dir="./optimized_reranker_results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4, # LoRA generally requires slightly higher learning rates than full tuning
    fp16=True,
    bf16=False, # Recommended for numerical stability when using bfloat16 compute dtype
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch"
)

# Instantiate the standard Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

In [ ]:
# @title Run the fine-tuning

trainer.train()